# Frequency Data Runs for the Olmo 2 Family
1B, 7B and 13B

Experiments are run on Colab. Change the model name variable and the revision for each separate run.

### Olmo 2 - 1B
model_name = OLMo-2-0425-1B  
revision = stage1-step990000-tokens2077B

### Olmo 2 - 7B
model_name = OLMo-2-1124-7B  
revision = stage1-step99000-tokens416B

### Olmo 2 - 13B
model_name = OLMo-2-1124-13B  
revision = stage1-step99000-tokens831B

## Setup

In [ ]:
# Imports
from google.colab import userdata
from pathlib import Path
import os

import sys
import torch
import time
import pandas as pd

import importlib

In [ ]:
# setup and import repo
token = userdata.get("GH_TMLR")

repo_owner = "trishasalas"
repo_name = "tmlr"

PROJECT_ROOT = Path("/content") / repo_name
repo_url = f"https://{token}@github.com/{repo_owner}/{repo_name}.git"
if not PROJECT_ROOT.exists():
    !git clone {repo_url} {PROJECT_ROOT}

os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

Project root: /content/tmlr


In [28]:
#  Device check
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: cuda


In [52]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "OLMo-2-0425-1B"
revision = "stage1-step990000-tokens2077B"

tokenizer = AutoTokenizer.from_pretrained(
    f"allenai/{model_name}",
    revision=revision,
)

olmo = AutoModelForCausalLM.from_pretrained(
    f"allenai/{model_name}",
    revision=revision,
)

config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.34k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/14.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

In [53]:
olmo = olmo.to("cuda")
olmo.eval()

print(next(olmo.parameters()).device)
print(next(olmo.parameters()).dtype)

cuda:0
torch.float32


In [54]:
olmo.eval()
olmo.config.use_cache = True

### Prompts

In [55]:
print("GPU:", torch.cuda.get_device_name(0))
print("Model device:", next(olmo.parameters()).device)
print("Model dtype:", next(olmo.parameters()).dtype)
print("Device map:", getattr(olmo, "hf_device_map", None))
print("Cache:", olmo.config.use_cache)

GPU: NVIDIA A100-SXM4-80GB
Model device: cuda:0
Model dtype: torch.float32
Device map: None
Cache: True


In [56]:
from src.elicitation import run_all_prompts
class HFModelWrapper:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    @torch.inference_mode()
    def generate(self, prompt, max_new_tokens=50, temperature=0):
        device = next(self.model.parameters()).device

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
        ).to(device)

        output = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
            pad_token_id=self.tokenizer.eos_token_id,
        )

        return self.tokenizer.decode(
            output[0],
            skip_special_tokens=True,
        )


olmo_wrapped = HFModelWrapper(olmo, tokenizer)

olmo_wrapped = HFModelWrapper(olmo, tokenizer)
results_df = run_all_prompts(olmo_wrapped, model_name, PROJECT_ROOT)
results_df

Saved 92 results to /content/tmlr/results/gpt2/OLMo-2-0425-1B-results.csv


,prompt_id,concept,prompt_type,template_type,prompt,output,max_tokens,model
0,decl_screen_reader_001,screen reader,declarative,cloze,A screen reader is,a device that is used to access the internet a...,100,OLMo-2-0425-1B
1,decl_wcag_001,WCAG,declarative,cloze,WCAG stands for,"the ""Web Content Accessibility Guidelines"" and...",100,OLMo-2-0425-1B
2,decl_skip_link_001,skip link,declarative,cloze,A skip link is,a link that is not a link. It is a link that i...,100,OLMo-2-0425-1B
3,decl_alt_text_001,alt text,declarative,cloze,The purpose of alt text is,to describe the image. It is not to describe t...,100,OLMo-2-0425-1B
4,decl_aria_001,ARIA,declarative,cloze,ARIA stands for,"the ""European Union of the Mediterranean Sea"" ...",100,OLMo-2-0425-1B
...,...,...,...,...,...,...,...,...
87,ctrl_bicycle_cloze_001,bicycle,control,cloze,A bicycle is used for,"transportation, and it is a very important par...",100,OLMo-2-0425-1B
88,ctrl_bicycle_direct_001,bicycle,control,direct_question,What is a bicycle?,A bicycle is a vehicle that is propelled by a ...,100,OLMo-2-0425-1B
89,ctrl_bicycle_instruction_001,bicycle,control,instruction,Explain bicycles to a web developer.,1. What is a bicycle?\n 2. What is a bicycle ...,100,OLMo-2-0425-1B
90,ctrl_bicycle_evaluative_001,bicycle,control,evaluative,A bicycle without brakes is not safe because,it is not possible to stop the bike in case of...,100,OLMo-2-0425-1B


In [57]:
from src.frequency import build_frequency_table
dolma_freq = build_frequency_table(index="v4_olmo-mix-1124_llama")

Querying: screen_reader...
Querying: alt_text...
Querying: skip_link...
Querying: color_contrast...
Querying: keyboard_navigation...
Querying: focus_indicator...
Querying: semantic_html...
Querying: closed_captions...
Querying: keyboard_interaction...
Querying: section_heading...
Querying: text_alternative...
Querying: audio_description...
Querying: sign_language...
Querying: sensory_characteristics...
Querying: input_purpose...
Querying: target_size...
Querying: touch_target...
Querying: drag_movement...
Querying: focus_appearance...
Querying: consistent_help...
Querying: redundant_entry...
Querying: accessible_authentication...
Querying: text_spacing...
Querying: status_message...
Querying: error_identification...
Querying: pointer_cancellation...
Querying: character_key...
  Rate limited on 'character key', waiting 2s (attempt 1/5)
  Rate limited on 'character key', waiting 4s (attempt 2/5)
  Rate limited on 'character key', waiting 8s (attempt 3/5)
Querying: accessibility_tree...
Q

In [58]:
from src.accuracy_coding import code_response

results_df['accuracy'] = results_df.apply(
    lambda r: code_response(r['prompt_type'], r['concept'], r['prompt'], r['output']),
    axis=1
)

In [59]:
score_map = {'correct': 1.0, 'partial': 0.5, 'incorrect': 0.0}
results_df['score'] = results_df['accuracy'].map(score_map)

compound_accuracy = results_df.groupby('concept').agg(
    mean_accuracy=('score', 'mean'),
    n=('score', 'count')
).reset_index()
compound_accuracy['compound'] = compound_accuracy['concept'].str.replace(' ', '_')
compound_accuracy['compound'] = compound_accuracy['compound'].str.lower()

In [60]:
import numpy as np
from scipy.stats import spearmanr

merged = compound_accuracy.merge(dolma_freq, on='compound', how='inner')
merged['log_freq'] = np.log10(merged['bigram_count'].clip(lower=1))

rho, p = spearmanr(merged['log_freq'], merged['mean_accuracy'])
print(f"{model_name} Spearman: ρ={rho:.4f}, p={p:.4f}, n={len(merged)}")

OLMo-2-0425-1B Spearman: ρ=0.3288, p=0.0211, n=49


In [61]:
import numpy as np
from scipy.stats import spearmanr

dtype = next(olmo.parameters()).device

merged = compound_accuracy.merge(dolma_freq, on='compound', how='inner')
merged['log_freq'] = np.log10(merged['bigram_count'].clip(lower=1))

rho, p = spearmanr(merged['log_freq'], merged['mean_accuracy'])
print(f"{model_name} Spearman: ρ={rho:.4f}, p={p:.4f}, n={len(merged)}")

# Save results
merged.to_csv(f'{model_name.replace("/", "_")}_spearman_merged.csv', index=False)

with open(f'{model_name.replace("/", "_")}_spearman_result.md', 'w') as f:
  f.write(f"# Spearman Result: {model_name}\n")
  f.write(f"Revision: {revision}\n")
  f.write(f"Model dtype: {next(olmo.parameters()).dtype}\n")
  f.write(f"- ρ = {rho:.4f}\n")
  f.write(f"- p = {p:.4f}\n")
  f.write(f"- n = {len(merged)}\n")
  f.write(f"- corpus: OLMo-Mix-1124\n")
  f.write(f"- index: v4_olmo-mix-1124_llama\n\n")

print(f"Saved to {model_name.replace('/', '_')}_spearman_result.md")

OLMo-2-0425-1B Spearman: ρ=0.3288, p=0.0211, n=49
Saved to OLMo-2-0425-1B_spearman_result.md


In [ ]:
# Cell 15b: Partial correlations — confound controls
# Same method as dual_spearman.py partial for Pythia/GPT-2

def _partial_spearman(x, y, z):
    """Spearman partial correlation of (x, y) controlling z."""
    rx, ry, rz = (pd.Series(v).rank() for v in (x, y, z))
    rxy = np.corrcoef(rx, ry)[0, 1]
    rxz = np.corrcoef(rx, rz)[0, 1]
    ryz = np.corrcoef(ry, rz)[0, 1]
    denom = math.sqrt(max((1 - rxz**2) * (1 - ryz**2), 1e-12))
    return (rxy - rxz * ryz) / denom

# Token count per compound using OLMo tokenizer (still loaded)
merged['token_count'] = merged['compound'].apply(
    lambda c: len(tokenizer.tokenize(c.replace('_', ' ')))
)

partial_rows = []
for zcol, label in [('token_count', 'compound_token_count'),
                     ('word1_count', 'word1_unigram_count')]:
    pr = _partial_spearman(merged['log_freq'], merged['mean_accuracy'], merged[zcol])
    partial_rows.append({
        'suite': model_name,
        'control': label,
        'partial_rho': round(pr, 4),
        'raw_rho': round(rho, 4),
        'n_compounds': len(merged)
    })
    print(f"  Partial ρ (controlling {label}): {pr:.4f}  (raw: {rho:.4f})")

# Save alongside existing results
partial_df = pd.DataFrame(partial_rows)
partial_df.to_csv(f'{model_name.replace("/", "_")}_spearman_partial.csv', index=False)

# Append to the result markdown
with open(f'{model_name.replace("/", "_")}_spearman_result.md', 'a') as f:
    f.write(f"## Partial Correlations\n")
    for _, row in partial_df.iterrows():
        f.write(f"- Controlling {row['control']}: partial ρ = {row['partial_rho']}\n")
    f.write(f"\n")

print(f"Saved to {model_name.replace('/', '_')}_spearman_partial.csv")

### Delete Model & Clear Cache

In [62]:
# Free memory for next model
import gc
del olmo
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"Memory cleared — GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB allocated")
elif device == "mps":
    torch.mps.empty_cache()
    print("Memory cleared")
!rm -rf ~/.cache/huggingface

Memory cleared — GPU: 5.9GB allocated
